# Dataset Collection Disease Split Workflow

This notebook retrieves dataset entities from a Synapse `DatasetCollection`, reads current dataset-level annotations, splits `disease` values into `disease` and `diseaseSubtype`, previews all changes, and optionally applies updates after verification.

Notes:
- Scope is dataset-level annotations only (not file annotations).
- `dataset_code` updates are intentionally out of scope.
- Apply step uses the same `entity.annotations` + `syn.store(entity)` pattern used in `synapse_dataset_manager.py`.

In [7]:
import ast
import json
import os
import re
from datetime import datetime
from typing import Any, Dict, List, Tuple

import pandas as pd
from IPython.display import display
import synapseclient
from synapseclient.models import Dataset, DatasetCollection


In [15]:
# ==================== USER CONFIG ====================
COLLECTION_ID = "syn66496326"  # required
DRY_RUN = False # set to True to only generate the preview CSV without applying changes
APPLY_AFTER_VERIFICATION = True # set to True to apply changes after verifying the preview CSV. Make sure to update CSV_SOURCE_FOR_APPLY if you edited the preview CSV and want to apply those changes instead of the original preview CSV.

# Optional: explicit disease string -> (disease, diseaseSubtype) mapping
# Values can be strings or lists. Empty subtype is allowed.
CUSTOM_SPLIT_MAP: Dict[str, Dict[str, Any]] = {
    # Example:
    # "ALS: C9orf72": {"disease": ["ALS"], "diseaseSubtype": ["C9orf72"]},
}

OUTPUT_DIR = "../annotations"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TIMESTAMP = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
PREVIEW_CSV = os.path.join(OUTPUT_DIR, f"collection_{COLLECTION_ID}_disease_split_preview_{TIMESTAMP}.csv")
CSV_SOURCE_FOR_APPLY = PREVIEW_CSV  # set this to your edited CSV path before apply
BACKUP_JSON = os.path.join(OUTPUT_DIR, f"collection_{COLLECTION_ID}_disease_split_backup_{TIMESTAMP}.json")

print(f"Collection: {COLLECTION_ID}")
print(f"DRY_RUN: {DRY_RUN}")
print(f"APPLY_AFTER_VERIFICATION: {APPLY_AFTER_VERIFICATION}")
print(f"Preview CSV: {PREVIEW_CSV}")
print(f"CSV source for apply: {CSV_SOURCE_FOR_APPLY}")
print(f"Backup JSON: {BACKUP_JSON}")

Collection: syn66496326
DRY_RUN: False
APPLY_AFTER_VERIFICATION: True
Preview CSV: ../annotations/collection_syn66496326_disease_split_preview_20260216_160114.csv
CSV source for apply: ../annotations/collection_syn66496326_disease_split_preview_20260216_160114.csv
Backup JSON: ../annotations/collection_syn66496326_disease_split_backup_20260216_160114.json


In [16]:
def connect_to_synapse():
    """Connect using Synapse default credential discovery or environment token."""
    syn = synapseclient.Synapse()
    auth_token = os.getenv("SYNAPSE_AUTH_TOKEN")
    if auth_token:
        syn.login(authToken=auth_token)
        print("Connected to Synapse using SYNAPSE_AUTH_TOKEN")
    else:
        syn.login()
        print("Connected to Synapse using default credentials")
    return syn


def list_dataset_ids_from_collection(collection_id: str) -> List[str]:
    """Return dataset IDs contained in a DatasetCollection."""
    collection = DatasetCollection(id=collection_id).get()
    items = getattr(collection, "items", [])
    dataset_ids = []
    for item in items:
        item_id = getattr(item, "id", None)
        if item_id and str(item_id).startswith("syn"):
            dataset_ids.append(str(item_id))
    return dataset_ids


def normalize_to_list(value: Any) -> List[str]:
    """Normalize scalar/list annotation values to a clean list of strings."""
    if value is None:
        return []
    if isinstance(value, list):
        out = []
        for v in value:
            if v is None:
                continue
            text = str(v).strip()
            if text:
                out.append(text)
        return out
    text = str(value).strip()
    return [text] if text else []


def dedupe_keep_order(values: List[str]) -> List[str]:
    seen = set()
    out = []
    for v in values:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out


CANONICAL_DISEASE_ALIASES = {
    "ALS": "ALS",
    "AMYOTROPHIC LATERAL SCLEROSIS": "ALS",
    "FTD": "FTD",
    "FRONTOTEMPORAL DEMENTIA": "FTD",
    "Alzheimer's Disease": "Alzheimer's Disease",
    "Parkinson's Disease": "Parkinson's Disease",
    "Other Neurological Disorders": "Other Neurological Disorders",
    "Other Motor Neuron Disease": "Other Motor Neuron Disease"
}


def canonicalize_disease_token(value: str) -> str:
    normalized = re.sub(r"\s+", " ", value.strip()).upper()
    return CANONICAL_DISEASE_ALIASES.get(normalized, "")


def parse_disease_value(value: str) -> Tuple[List[str], List[str], bool]:
    """
    Fallback parser for disease values.
    Returns (disease_list, subtype_list, parsed_successfully).
    """
    text = value.strip()
    normalized_upper = re.sub(r"\s+", " ", text).upper()
    if not text:
        return [], [], True

    canonical = canonicalize_disease_token(text)
    if canonical:
        return [canonical], [], True

    # ALS/FTD subtype labels should be preserved as subtype values.
    if normalized_upper.endswith("-ALS") or (normalized_upper.endswith(" ALS") and normalized_upper != "AMYOTROPHIC LATERAL SCLEROSIS"):
        return ["ALS"], [text], True
    if normalized_upper.endswith("-FTD") or (normalized_upper.endswith(" FTD") and normalized_upper != "FRONTOTEMPORAL DEMENTIA"):
        return ["FTD"], [text], True

    for delim in [" : ", ":", " - ", ";", "|"]:
        if delim in text:
            left, right = text.split(delim, 1)
            left = left.strip()
            right = right.strip()
            if left and right:
                left_canonical = canonicalize_disease_token(left)
                if left_canonical:
                    return [left_canonical], [right], True
                right_canonical = canonicalize_disease_token(right)
                if right_canonical:
                    return [right_canonical], [left], True
                return [left], [right], True

    # Pattern: Something (Subtype)
    m = re.match(r"^(.*?)\s*\((.*?)\)\s*$", text)
    if m:
        base = m.group(1).strip()
        subtype = m.group(2).strip()
        if base and subtype:
            base_canonical = canonicalize_disease_token(base)
            if base_canonical:
                return [base_canonical], [subtype], True
            return [base], [subtype], True

    return [text], [], False


def split_disease_values(values: List[str], custom_map: Dict[str, Dict[str, Any]]) -> Tuple[List[str], List[str], List[str]]:
    """Split normalized disease values into disease + diseaseSubtype with unresolved tracking."""
    diseases: List[str] = []
    subtypes: List[str] = []
    unresolved: List[str] = []

    for raw in values:
        if raw in custom_map:
            mapped = custom_map[raw]
            diseases.extend(normalize_to_list(mapped.get("disease")))
            subtypes.extend(normalize_to_list(mapped.get("diseaseSubtype")))
            continue

        parsed_disease, parsed_subtype, ok = parse_disease_value(raw)
        diseases.extend(parsed_disease)
        subtypes.extend(parsed_subtype)
        if not ok:
            unresolved.append(raw)

    return (
        dedupe_keep_order(diseases),
        dedupe_keep_order(subtypes),
        dedupe_keep_order(unresolved),
    )


def safe_annotations(entity) -> Dict[str, Any]:
    if hasattr(entity, "annotations") and entity.annotations:
        return dict(entity.annotations)
    return {}


def apply_dataset_annotations(syn, dataset_id: str, proposed: Dict[str, Any], dry_run: bool = True) -> Dict[str, Any]:
    """Apply disease + diseaseSubtype on dataset entity and verify persisted values."""
    result = {"dataset_id": dataset_id, "status": "pending", "error": None}
    try:
        entity = syn.get(dataset_id, downloadFile=False)
        current = safe_annotations(entity)

        current["disease"] = proposed.get("disease", [])
        subtype_vals = normalize_to_list(proposed.get("diseaseSubtype"))
        if subtype_vals:
            current["diseaseSubtype"] = subtype_vals
        elif "diseaseSubtype" in current:
            del current["diseaseSubtype"]

        if dry_run:
            result["status"] = "dry_run"
            return result

        entity.annotations = current
        syn.store(entity)

        # Verify persistence
        verify_entity = syn.get(dataset_id, downloadFile=False)
        verify_ann = safe_annotations(verify_entity)
        expected_disease = normalize_to_list(proposed.get("disease"))
        actual_disease = normalize_to_list(verify_ann.get("disease"))
        expected_subtype = normalize_to_list(proposed.get("diseaseSubtype"))
        actual_subtype = normalize_to_list(verify_ann.get("diseaseSubtype"))

        if expected_disease == actual_disease and expected_subtype == actual_subtype:
            result["status"] = "applied"
        else:
            result["status"] = "verify_mismatch"
            result["error"] = (
                f"Expected disease={expected_disease}, diseaseSubtype={expected_subtype}; "
                f"got disease={actual_disease}, diseaseSubtype={actual_subtype}"
            )

        return result
    except Exception as e:
        result["status"] = "error"
        result["error"] = str(e)
        return result


In [17]:
# ==================== FETCH + PREPARE CHANGES ====================
syn = connect_to_synapse()
dataset_ids = list_dataset_ids_from_collection(COLLECTION_ID)
print(f"Found {len(dataset_ids)} datasets in collection {COLLECTION_ID}")

records = []
backup_records = []

for dataset_id in dataset_ids:
    entity = syn.get(dataset_id, downloadFile=False)
    name = getattr(entity, "name", dataset_id)
    original_annotations = safe_annotations(entity)

    original_disease = normalize_to_list(original_annotations.get("disease"))
    original_subtype = normalize_to_list(original_annotations.get("diseaseSubtype"))

    new_disease, new_subtype, unresolved = split_disease_values(original_disease, CUSTOM_SPLIT_MAP)

    changed = (
        original_disease != new_disease
        or original_subtype != new_subtype
    )

    proposed_annotations = dict(original_annotations)
    proposed_annotations["disease"] = new_disease
    if new_subtype:
        proposed_annotations["diseaseSubtype"] = new_subtype
    elif "diseaseSubtype" in proposed_annotations:
        del proposed_annotations["diseaseSubtype"]

    records.append({
        "dataset_id": dataset_id,
        "dataset_name": name,
        "original_disease": original_disease,
        "proposed_disease": new_disease,
        "original_diseaseSubtype": original_subtype,
        "proposed_diseaseSubtype": new_subtype,
        "changed": changed,
        "unresolved_values": unresolved,
    })

    backup_records.append({
        "dataset_id": dataset_id,
        "dataset_name": name,
        "original_annotations": original_annotations,
        "proposed_annotations": proposed_annotations,
        "changed": changed,
        "unresolved_values": unresolved,
        "apply_status": "pending",
        "post_apply_annotations": None,
    })

preview_df = pd.DataFrame(records)
preview_df.to_csv(PREVIEW_CSV, index=False)

backup_payload = {
    "collection_id": COLLECTION_ID,
    "generated_at_utc": datetime.utcnow().isoformat(),
    "dry_run": DRY_RUN,
    "apply_after_verification": APPLY_AFTER_VERIFICATION,
    "records": backup_records,
}
with open(BACKUP_JSON, "w") as f:
    json.dump(backup_payload, f, indent=2)

total = len(preview_df)
changed_count = int(preview_df["changed"].sum()) if total else 0
unresolved_count = int(preview_df["unresolved_values"].apply(lambda x: len(x) > 0).sum()) if total else 0

print(f"Preview generated: {PREVIEW_CSV}")
print(f"Backup JSON generated: {BACKUP_JSON}")
print(f"Datasets scanned: {total}")
print(f"Datasets changed: {changed_count}")
print(f"Datasets with unresolved values: {unresolved_count}")

preview_df.head(20)

Welcome, ram.ayyala!

Connected to Synapse using default credentials


[WARNING] /tmp/ipykernel_136816/769326165.py:10: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  entity = syn.get(dataset_id, downloadFile=False)



Found 22 datasets in collection syn66496326
Preview generated: ../annotations/collection_syn66496326_disease_split_preview_20260216_160114.csv
Backup JSON generated: ../annotations/collection_syn66496326_disease_split_backup_20260216_160114.json
Datasets scanned: 22
Datasets changed: 17
Datasets with unresolved values: 9


,dataset_id,dataset_name,original_disease,proposed_disease,original_diseaseSubtype,proposed_diseaseSubtype,changed,unresolved_values
0,syn67751280,Loss of Nuclear TDP-43 Is Associated with Deco...,"[FTD, FTD-ALS]","[FTD, ALS]",[],[FTD-ALS],True,[]
1,syn67748058,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, C9-FTD, Control]","[ALS, FTD, Control]",[],"[C9-ALS, C9-FTD]",True,[Control]
2,syn67746133,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, C9-FTD, Control]","[ALS, FTD, Control]",[],"[C9-ALS, C9-FTD]",True,[Control]
3,syn72016774,Trehalose Biomarker Dataset,[ALS],[ALS],[],[],False,[]
4,syn67729513,Unexpected similarities between C9ORF72 and sp...,"[ALS, FTD, Control, Pre-fALS, Parkinson's Dise...","[ALS, FTD, Control, Pre-fALS, Parkinson's Dise...",[],"[PD, AD]",True,"[Control, Pre-fALS]"
5,syn67719824,Neurovascular dysfunction in GRN-associated fr...,"[Control, FTD]","[Control, FTD]",[],[],False,[Control]
6,syn72379204,A Study to Evaluate Efficacy Safety and Tolera...,[ALS],[ALS],[Unknown],[],True,[]
7,syn72379205,A Study to Evaluate the Efficacy and Safety of...,[ALS],[ALS],[Unknown],[],True,[]
8,syn67740891,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, Control]","[ALS, Control]",[],[C9-ALS],True,[Control]
9,syn72379206,ALS Analysis and Assessment Survey Study,[ALS],[ALS],[Unknown],[],True,[]


## Verification Checkpoint

Review the preview table and CSV before applying updates.

Set both values below before running the apply cell:
- `DRY_RUN = False`
- `APPLY_AFTER_VERIFICATION = True`

In [18]:
# Optional: focus review on changed/unresolved records
changed_df = preview_df[preview_df["changed"] == True].copy()
unresolved_df = preview_df[preview_df["unresolved_values"].apply(lambda x: len(x) > 0)].copy()

print(f"Changed rows: {len(changed_df)}")
print(f"Unresolved rows: {len(unresolved_df)}")

display(changed_df.head(50))
display(unresolved_df.head(50))

Changed rows: 17
Unresolved rows: 9


,dataset_id,dataset_name,original_disease,proposed_disease,original_diseaseSubtype,proposed_diseaseSubtype,changed,unresolved_values
0,syn67751280,Loss of Nuclear TDP-43 Is Associated with Deco...,"[FTD, FTD-ALS]","[FTD, ALS]",[],[FTD-ALS],True,[]
1,syn67748058,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, C9-FTD, Control]","[ALS, FTD, Control]",[],"[C9-ALS, C9-FTD]",True,[Control]
2,syn67746133,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, C9-FTD, Control]","[ALS, FTD, Control]",[],"[C9-ALS, C9-FTD]",True,[Control]
4,syn67729513,Unexpected similarities between C9ORF72 and sp...,"[ALS, FTD, Control, Pre-fALS, Parkinson's Dise...","[ALS, FTD, Control, Pre-fALS, Parkinson's Dise...",[],"[PD, AD]",True,"[Control, Pre-fALS]"
6,syn72379204,A Study to Evaluate Efficacy Safety and Tolera...,[ALS],[ALS],[Unknown],[],True,[]
7,syn72379205,A Study to Evaluate the Efficacy and Safety of...,[ALS],[ALS],[Unknown],[],True,[]
8,syn67740891,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, Control]","[ALS, Control]",[],[C9-ALS],True,[Control]
9,syn72379206,ALS Analysis and Assessment Survey Study,[ALS],[ALS],[Unknown],[],True,[]
10,syn67754663,Loss of Nuclear TDP-43 Is Associated with Deco...,"[FTD, FTD-ALS]","[FTD, ALS]",[],[FTD-ALS],True,[]
12,syn67713129,Abnormal RNA stability in amyotrophic lateral ...,"[Control, C9-ALS, sporadic ALS]","[Control, ALS]",[],"[C9-ALS, sporadic ALS]",True,[Control]


,dataset_id,dataset_name,original_disease,proposed_disease,original_diseaseSubtype,proposed_diseaseSubtype,changed,unresolved_values
1,syn67748058,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, C9-FTD, Control]","[ALS, FTD, Control]",[],"[C9-ALS, C9-FTD]",True,[Control]
2,syn67746133,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, C9-FTD, Control]","[ALS, FTD, Control]",[],"[C9-ALS, C9-FTD]",True,[Control]
4,syn67729513,Unexpected similarities between C9ORF72 and sp...,"[ALS, FTD, Control, Pre-fALS, Parkinson's Dise...","[ALS, FTD, Control, Pre-fALS, Parkinson's Dise...",[],"[PD, AD]",True,"[Control, Pre-fALS]"
5,syn67719824,Neurovascular dysfunction in GRN-associated fr...,"[Control, FTD]","[Control, FTD]",[],[],False,[Control]
8,syn67740891,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, Control]","[ALS, Control]",[],[C9-ALS],True,[Control]
11,syn67737779,Postmortem Cortex Samples Identify Distinct Mo...,"[ALS, FTD, Control, Pre-fALS, Alzheimer's Dise...","[ALS, FTD, Control, Pre-fALS, Alzheimer's Dise...",[],[],False,"[Control, Pre-fALS, Alzheimer's Disease, Other..."
12,syn67713129,Abnormal RNA stability in amyotrophic lateral ...,"[Control, C9-ALS, sporadic ALS]","[Control, ALS]",[],"[C9-ALS, sporadic ALS]",True,[Control]
13,syn67743547,Divergent impacts of C9orf72 repeat expansion ...,"[C9-ALS, C9-FTD, Control]","[ALS, FTD, Control]",[],"[C9-ALS, C9-FTD]",True,[Control]
16,syn67733559,Truncated stathmin-2 is a marker of TDP-43 pat...,"[ALS, FTD, Control, Pre-fALS, Alzheimer's Dise...","[ALS, FTD, Control, Pre-fALS, Alzheimer's Dise...",[],[],False,"[Control, Pre-fALS, Alzheimer's Disease, Parki..."


In [19]:
# ==================== APPLY CHANGES ====================
if DRY_RUN or not APPLY_AFTER_VERIFICATION:
    print("Apply skipped. Set DRY_RUN=False and APPLY_AFTER_VERIFICATION=True after review.")
else:
    apply_results = []
    changed_ids = set(preview_df[preview_df["changed"] == True]["dataset_id"].tolist())

    for rec in backup_payload["records"]:
        dataset_id = rec["dataset_id"]
        if dataset_id not in changed_ids:
            rec["apply_status"] = "unchanged"
            continue

        result = apply_dataset_annotations(
            syn=syn,
            dataset_id=dataset_id,
            proposed=rec["proposed_annotations"],
            dry_run=False,
        )

        rec["apply_status"] = result.get("status", "unknown")
        if result.get("error"):
            rec["apply_error"] = result.get("error")

        # Capture post-apply annotations for audit
        try:
            post_entity = syn.get(dataset_id, downloadFile=False)
            rec["post_apply_annotations"] = safe_annotations(post_entity)
        except Exception as e:
            rec["post_apply_annotations"] = {"error": str(e)}

        apply_results.append(result)

    backup_payload["applied_at_utc"] = datetime.utcnow().isoformat()
    with open(BACKUP_JSON, "w") as f:
        json.dump(backup_payload, f, indent=2)

    results_df = pd.DataFrame(apply_results)
    if len(results_df) == 0:
        print("No changed datasets to apply.")
    else:
        print(results_df["status"].value_counts(dropna=False))
        display(results_df)

    print(f"Updated backup JSON with apply results: {BACKUP_JSON}")

[WARNING] /tmp/ipykernel_136816/1862356936.py:154: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  entity = syn.get(dataset_id, downloadFile=False)

[WARNING] /tmp/ipykernel_136816/1862356936.py:169: DeprecationWarning: Call to deprecated method store. (To be removed in 5.0.0. Use `from synapseclient.operations import store` instead.) -- Deprecated since version 4.11.0.
  syn.store(entity)

[WARNING] /tmp/ipykernel_136816/1862356936.py:172: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since version 4.11.0.
  verify_entity = syn.get(dataset_id, downloadFile=False)

[WARNING] /tmp/ipykernel_136816/775652107.py:27: DeprecationWarning: Call to deprecated method get. (To be removed in 5.0.0. Use `from synapseclient.operations import get` instead.) -- Deprecated since ve

status
applied    17
Name: count, dtype: int64


,dataset_id,status,error
0,syn67751280,applied,None
1,syn67748058,applied,None
2,syn67746133,applied,None
3,syn67729513,applied,None
4,syn72379204,applied,None
5,syn72379205,applied,None
6,syn67740891,applied,None
7,syn72379206,applied,None
8,syn67754663,applied,None
9,syn67713129,applied,None


Updated backup JSON with apply results: ../annotations/collection_syn66496326_disease_split_backup_20260216_160114.json
